In [ ]:
%%html

<style>
    .container-fluid.main-container {
        margin-left: 3em;
        margin-right: 8em;
        max-width: 100vw;
    }
</style>

# Introduction

This notebook contains the implementation of a partially spatially coherent propagation algorithm based on GVS, modified to model a projector's light-scanning process. We treat this process as a partially spatially coherent wavefront arriving at the SLM plane, with a coherence area proportional to the projected spot size (or the projected pixel pitch). This allows us to construct an efficient forward model based on the GVS formulation rather than propagating each point source in the SLM plane independently. We then use stochastic gradient descent with AdamW in PyTorch to jointly generate intensity and phase holograms in an inverse-problem fashion.

<center><img alt="Experimental Setup" src="img/optical-system.jpeg" width="512"/></center>

The intensity $I$ that reaches the camera plane after the laser scanning process can be modeled as an incoherent sum of sequential coherent propagations of the phase hologram $h (x, y)$, having the laser beam spot $b (x, y)$ centered on each SLM pixel, weighted by the intensity hologram $A$,
$$
I = \sum_{n=1}^N \sum_{m=1}^M A_{nm}
    \left | \mathcal{P}\{ b_s(x, y) \ \exp [2 \pi j \ h(x, y)] \} \right |^2
\qquad (1)
$$

A better solution is to use a local scaling approximation to express $I$ as a convolution between the light source intensity distribution ($I_S$) and the coherent propagated intensity of the modulated wavefront at the SLM plane ($I_C$), closely following the GVS model,
$$
I = \mathcal{F}^{-1} \left\{
    \mathcal{F}\{ I_C \}
    \mathcal{F}\{ I_{S\textrm{-norm}} \}
    \right \}
\qquad (2)
$$

In [ ]:
import numpy as np
import plotly
import plotly.express as px
import torch

from time import perf_counter
from torchvision.io import read_image, ImageReadMode

from gvs_propagator.utils import crop_center, create_grid, normalize, psnr
from gvs_propagator.constants import nm, um, mm
from gvs_propagator.cgh.projector import (
    resize,
    zero_pad,
    lsa_propagate,
    CoherentOptimizer,
    PhaseOnlyOptimizer,
    IntensityPhaseOptimizer,
)

plotly.offline.init_notebook_mode()

# Numpy Implementation

In [ ]:
nx = 128
dx = 7 * um
wavelength = 532 * nm
z = 7 * mm
k = 2 * np.pi / wavelength
laser_spot_sigma = 3 * dx
epsilon = 1e-12

In [ ]:
# Setup
x, y = create_grid(nx, dx)
r2 = x ** 2 + y ** 2

df = 1 / (nx * dx)
f = (np.arange(-nx / 2, nx / 2)) * df
fx, fy = np.meshgrid(f, f)
fresnel_transfer_function = np.fft.ifftshift(np.exp(1j * k * z - 1j * np.pi * wavelength * z * (fx ** 2 + fy ** 2)))

square_mask = (np.abs(x) < 40 * dx) & (np.abs(y) < 40 * dx)
phase = square_mask * np.exp(2j * np.pi * np.random.RandomState(42).rand(*square_mask.shape))
intensity_modulation = 0.5 + 0.5 * np.cos(2 * np.pi * x / (5 * dx)) * np.cos(2 * np.pi * y / (5 * dx))

# Exhaustive Brute-Force Scanning (Reference, Eq. 1)
start = perf_counter()

laser_intensity = np.exp(-r2 / (2 * laser_spot_sigma ** 2))
intensity_exhaustive = np.zeros((nx, nx), dtype=np.float64)
for iy in range(nx):
    for ix in range(nx):
        shift_x = ix - nx // 2
        shift_y = iy - nx // 2

        laser_intensity_shifted = np.roll(np.roll(laser_intensity, shift_y, axis=0), shift_x, axis=1)
        field_slm = laser_intensity_shifted * phase
        field_camera = np.fft.ifft2(np.fft.fft2(field_slm) * fresnel_transfer_function)
        intensity_exhaustive += intensity_modulation[iy, ix] * np.abs(field_camera) ** 2

intensity_exhaustive = normalize(intensity_exhaustive)
time_exhaustive = perf_counter() - start

# Local Scaling Approximation (Efficient, Eq. 2)
start = perf_counter()

laser_intensity = normalize(abs(np.exp(-r2 / (2 * laser_spot_sigma ** 2))) ** 2)
a_eff = np.fft.fftshift(np.fft.ifft2(np.fft.fft2(intensity_modulation) * np.fft.fft2(laser_intensity)))
baseline_intensity = np.abs(np.fft.ifft2(np.fft.fft2(phase * np.sqrt(a_eff + epsilon)) * fresnel_transfer_function)) ** 2
far_field_analytical_beam = normalize(np.exp(-4 * np.pi ** 2 * laser_spot_sigma ** 2 * r2 / (wavelength * z) ** 2))
intensity_approx = normalize(abs(np.fft.fftshift(np.fft.ifft2(np.fft.fft2(baseline_intensity) * np.fft.fft2(far_field_analytical_beam)))))

time_approx = perf_counter() - start

intensity_difference = np.abs(intensity_approx - intensity_exhaustive)

## Intensity Modulation $A_{nm}$

Rapidly varying checkerboard pattern.

In [ ]:
fig = px.imshow(
    intensity_modulation,
    x=x[0],
    y=x[0],
    color_continuous_scale="jet",
)
fig.show()

## Phase hologram $h(x, y)$

Displayed on the SLM.

In [ ]:
fig = px.imshow(
    np.mod(np.angle(phase), 2 * np.pi),
    x=x[0],
    y=x[0],
    color_continuous_scale="icefire",
)
fig.show()

## Propagated Intensity $I$

Measured at the camera plane.

In [ ]:
fig = px.imshow(
    np.array([intensity_exhaustive, intensity_approx]),
    x=x[0],
    y=x[0],
    color_continuous_scale="jet",
    facet_col=0,
)
fig.layout.annotations[0]["text"] = f"Intensity Exhaustive ({time_exhaustive:.3f} s)"
fig.layout.annotations[1]["text"] = f"Intensity Approximation ({time_approx:.3f} s)"
fig.show()

In [ ]:
psnr_intensities = psnr(intensity_exhaustive, intensity_approx)
print(f"PSNR: {psnr_intensities:.1f} dB")

# 2D Inverse Optimization in Pytorch

In [ ]:
device = "mps"  # Use cuda or cpu depending on installation
torch.set_default_device(device)
dx_projection = 76 * um
image_path = "img/usaf_and_star.png"
target_image = (read_image(image_path, mode=ImageReadMode.GRAY)[0] / 255).numpy()

## Coherent Optimization

In [ ]:
phase = CoherentOptimizer(
    image_path,
    nx := 1024,
    ny := 600,
    dx_slm := 3.74 * um,
    propagation_distances := [30 * mm],
    wavelength := 532 * nm,
    lr := 0.0005,
    iterations := 500,
    batch_size:= 1,
    device=device,
).solve()

### Optimized Phase

Displayed on the SLM.

In [ ]:
px.imshow(np.mod(np.angle(phase.cpu().detach().numpy()), 2 * np.pi), color_continuous_scale="icefire")

In [ ]:
coherent_reconstruction = crop_center(lsa_propagate(
    zero_pad(phase), nx * 2, ny * 2, dx_slm, dx_slm, dx_projection, propagation_distances[0], wavelength
)).cpu().detach().numpy()
psnr_coherent = psnr(target_image, coherent_reconstruction)

## Phase-only Optimization

In [ ]:
phase = PhaseOnlyOptimizer(
    image_path,
    nx := 1024,
    ny := 600,
    dx_slm := 3.74 * um,
    dx_projection,
    propagation_distances := [30 * mm],
    wavelength := 532 * nm,
    lr := 0.001,
    iterations := 500,
    batch_size:= 1,
    device=device,
).solve()

### Optimized Phase

Displayed on the SLM.

In [ ]:
px.imshow(np.mod(np.angle(phase.cpu().detach().numpy()), 2 * np.pi), color_continuous_scale="icefire")

In [ ]:
phase_only_reconstruction = crop_center(lsa_propagate(
    zero_pad(phase), nx * 2, ny * 2, dx_slm, dx_slm, dx_projection, propagation_distances[0], wavelength
)).cpu().detach().numpy()
psnr_phase_only = psnr(target_image, phase_only_reconstruction)

## Intensity and Phase Optimization

In [ ]:
intensity, phase = IntensityPhaseOptimizer(
    image_path,
    nx := 1024,
    ny := 600,
    dx_slm := 3.74 * um,
    dx_projection,
    propagation_distances := [30 * mm],
    wavelength := 532 * nm,
    lr := 0.001,
    iterations := 500,
    batch_size:= 1,
    device=device,
).solve()

### Optimized Intensity

Projected into the SLM.

In [ ]:
px.imshow(intensity.cpu().detach().numpy(), color_continuous_scale="jet")

### Optimized Phase

Displayed on the SLM.

In [ ]:
px.imshow(np.mod(np.angle(phase.cpu().detach().numpy()), 2 * np.pi), color_continuous_scale="icefire")

In [ ]:
intensity = zero_pad(resize(intensity, nx, ny))
phase = zero_pad(phase)
int_phase_reconstruction = crop_center(lsa_propagate(
    phase, nx * 2, ny * 2, dx_slm, dx_slm, dx_projection, propagation_distances[0], wavelength, intensity
)).cpu().detach().numpy()
psnr_int_phase = psnr(target_image, int_phase_reconstruction)

## Reconstructions

In [ ]:
coherent_reconstruction = np.stack([
    coherent_reconstruction * 0,
    coherent_reconstruction,
    coherent_reconstruction / 4.5,
], axis=-1)
phase_only_reconstruction = np.stack([
    phase_only_reconstruction * 0,
    phase_only_reconstruction,
    phase_only_reconstruction / 4.5,
], axis=-1)
int_phase_reconstruction = np.stack([
    int_phase_reconstruction * 0,
    int_phase_reconstruction,
    int_phase_reconstruction / 4.5,
], axis=-1)

fig = px.imshow(
    np.array([
        coherent_reconstruction,
        phase_only_reconstruction,
        int_phase_reconstruction,
    ]),
    color_continuous_scale="jet",
    facet_col=0,
    facet_col_wrap=2,
    height=600,
)
fig.layout.annotations[1]["text"] = f"Coherent: {psnr_coherent:.1f} dB PSNR"
fig.layout.annotations[2]["text"] = f"Phase-Only: {psnr_phase_only:.1f} dB PSNR"
fig.layout.annotations[0]["text"] = f"Intensity and Phase: {psnr_int_phase:.1f} dB PSNR"
fig.show()